
# Tracing the OVRO&#8211;LWA type II burst &#8212; B&#233;zier curves or point-and-click &#8212; 18 January 2026

Trace the band-split type II radio burst in the **OVRO&#8211;LWA** dynamic spectrum with either
**B&#233;zier curves** (start, end and one/two anchor points) or the **classic point-and-click**
picker, then report the shock diagnostics through a grid of **electron-density models &#215; fold
numbers** with Monte-Carlo error bars.

**Chain.** tracing &#8594; frequency lanes $f(t)$ for the fundamental (F) and harmonic (H) bands,
each split into a **lower** and **upper** branch &#8594; density models $n_e(r)$ &#8594; height
$r(t)$, drift rate, band-split density jump $X=(f_U/f_L)^2$, Alfv&#233;n Mach number $M_A$, shock
speed/acceleration, exciter electron energy, Alfv&#233;n speed and coronal magnetic field $B$.
The reference-lane height-time is additionally fit three ways &#8212; polynomial, Gallagher (2003)
and Byrne (2013) &#8212; for comparison.

**Choices set in the config cell.** `TRACE_MODE` picks B&#233;zier vs point-and-click;
`LANES_TO_TRACE` lets you skip any lane; `DEMO_MODE` runs the whole thing on a synthetic band-split
type II with no widget (useful for a headless check).

**Density models (folds 1&#8211;4):** Newkirk (1961), Saito (1970), Leblanc, Dulk & Bougeret
(1998), Baumbach&#8211;Allen (Allen 1947), Mann, Warmuth, Vocks & Rouillard (2023, *A&amp;A* 679,
A64). **Field comparison:** Dulk & McLean (1978), Gopalswamy & Yashiro (2011), Mann et al. (2023).

## 0&#160;&#183;&#160;Imports, constants and configuration

In [ ]:

import os
import pickle
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib import colors

from scipy.signal import savgol_filter
from scipy.optimize import curve_fit
from scipy.integrate import cumulative_trapezoid

from astropy.io import fits
from astropy.time import Time
from astropy.constants import R_sun, m_e, m_p, e, eps0, mu0, c

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **k):
        return x

warnings.filterwarnings('ignore', category=RuntimeWarning)

# --- physical constants pulled from astropy (no hardcoding of the physics) ---
R_SUN_M    = R_sun.to('m').value
C_MS       = c.to('m/s').value
E_CHARGE_J = e.si.value
M_E        = m_e.value
M_P        = m_p.value
EPS0       = eps0.value
MU0        = mu0.value

# electron plasma frequency: f_p[Hz] = PLASMA_CONST * sqrt(Ne[cm^-3])  (~8.98e3)
PLASMA_CONST = (1 / (2 * np.pi)) * np.sqrt(1e6 * E_CHARGE_J ** 2 / (EPS0 * M_E))
print(f'PLASMA_CONST = {PLASMA_CONST:.4g}  (f_pe[Hz] = PLASMA_CONST * sqrt(Ne[cm^-3]))')

In [ ]:

# =========================== CONFIGURATION ===========================
DEMO_MODE  = True                # True -> synthetic self-test (no widget); False -> trace the real spectrum
TRACE_MODE = 'bezier'            # 'bezier' = Bezier curves ; 'manual' = classic point-and-click picker

# --- event / data ---
EVENT_DATE = '2026-01-18'
DATA_ROOT  = './'                                        # set to '/home/mnedal/data' on the DIAS server
LWA_FITS   = f'{DATA_ROOT}/{EVENT_DATE}/LWA/20260118.fits'
OUTDIR     = './type2_lwa_bezier_outputs'
os.makedirs(OUTDIR, exist_ok=True)

# --- tracing window on the dynamic spectrum ---
TYPEII_WINDOW = (pd.Timestamp(f'{EVENT_DATE}T17:55:00'), pd.Timestamp(f'{EVENT_DATE}T18:14:00'))
TYPEII_FLIM   = [16, 62]                                 # MHz, tracing band
RADIO_BKG_WINDOW = (pd.Timestamp(f'{EVENT_DATE}T16:30:00'), pd.Timestamp(f'{EVENT_DATE}T17:00:00'))
LWA_POL_PLANE = 0                                        # 0 = Stokes I (confirm plane order with the data provider)

# --- emission physics ---
HARM = {'F': 1, 'H': 2}          # harmonic number per band (F = fundamental, H = harmonic)
MU   = 1.27                      # mean molecular weight per electron (10% He corona)

# --- density-model grid ---
FOLDS = [1, 2, 3, 4]             # fold numbers applied to every model
REF_MODEL_NAME = 'Newkirk x2'    # model x fold used for the reference per-lane figure

# --- which lanes to trace (drop any tuple to SKIP that lane) ---
LANES_TO_TRACE = [('F', 'lower'), ('F', 'upper'), ('H', 'lower'), ('H', 'upper')]

# --- Bezier tracing (interactive slider placement, jittered-ensemble error) ---
BEZIER_ANCHORS   = {('F', 'lower'): 1, ('F', 'upper'): 1,   # 1 -> quadratic, 2 -> cubic (per lane)
                    ('H', 'lower'): 1, ('H', 'upper'): 1}
BEZIER_NUM_POINTS = 80           # samples along each Bezier curve
BEZIER_N_REPEATS  = 5            # jittered repeats per lane (repeat 0 is unperturbed)
BEZIER_JITTER     = 1.5          # control-point jitter sigma, in dynamic-spectrum index pixels
BEZIER_SEED       = 0

# --- classic point-and-click tracing (TRACE_MODE = 'manual') ---
N_REPS_MANUAL     = 5            # reps traced per lane with the picker

# --- Monte-Carlo + height-time fitting comparison ---
N_MC     = 100                   # MC draws per pass for the fit error
POLY_DEG = 2                     # polynomial degree for the height-time fits
N_BOOT   = 300                   # bootstrap refits for the Gallagher/Byrne error bands
DT_DERIV = 0.5                   # s, step of the central-difference derivatives

# display down-sampling for the big pcolormesh previews (keeps them responsive)
PREVIEW_MAX_TCOLS = 800

def save_fig(fig, name, dpi=300):
    path = os.path.join(OUTDIR, f'{name}.png')
    fig.savefig(path, dpi=dpi, bbox_inches='tight')
    print('saved', path)

print(f'DEMO_MODE = {DEMO_MODE}, TRACE_MODE = {TRACE_MODE!r}')
print(f'lanes to trace: {[f"{b}-{p}" for b, p in LANES_TO_TRACE]}')
print(f'type II window {TYPEII_WINDOW[0].time()}-{TYPEII_WINDOW[1].time()} UT, {TYPEII_FLIM} MHz')


## 1&#160;&#183;&#160;Load the OVRO&#8211;LWA dynamic spectrum

Reads the beam dynamic-spectrum FITS (`npol, 1, nfreq, ntime`; `SFREQ` in GHz, `UT` = integer MJD
+ ms-of-day), slices the type II window and band, and builds the **background-ratioed** tracing
layer (each channel divided by its quiet pre-flare median). Under `DEMO_MODE` a synthetic
band-split type II is fabricated instead, so the notebook runs without the file or a widget.

In [ ]:

def load_lwa_dyspec(path, pol_plane=0):
    """OVRO-LWA dynamic-spectrum FITS. Returns (DatetimeIndex, freq[MHz], spec[nfreq, ntime])."""
    with fits.open(path) as hdul:
        cube = hdul[0].data[:, 0, :, :]                     # (npol, nfreq, ntime)
        fmhz = hdul['SFREQ'].data['SFREQ'] * 1e3            # GHz -> MHz
        ut = hdul['UT'].data
        time_mjd = ut['mjd'] + ut['time'] / 86400000
    t_index = pd.DatetimeIndex(Time(time_mjd, format='mjd').to_datetime())
    return t_index, np.asarray(fmhz), np.asarray(cube[pol_plane])


def make_typeii_layer(t_index, freqs, spec, window, flim, bkg_window):
    """Background-ratioed spectrogram slice (spec / per-channel pre-flare median) over the
    tracing window and band. Returns (t[window], f[band], layer[nf, nt])."""
    sel_t = (t_index >= window[0]) & (t_index <= window[1])
    sel_f = (freqs >= flim[0]) & (freqs <= flim[1])
    bsel  = (t_index >= bkg_window[0]) & (t_index <= bkg_window[1])
    bkg = np.nanmedian(spec[np.ix_(sel_f, bsel)].astype(float), axis=1)[:, None]
    with np.errstate(all='ignore'):
        layer = spec[np.ix_(sel_f, sel_t)].astype(float) / bkg
    return t_index[sel_t], freqs[sel_f], layer


def synth_typeii_layer(window, flim, cadence_s=0.514):
    """Fabricate a band-split type II for DEMO_MODE: two drifting F branches (lower/upper) plus
    their harmonic (x2 in frequency). Known drift and density jump so the chain can be checked."""
    t = pd.date_range(window[0], window[1], freq=f'{cadence_s}s')
    f = np.arange(flim[0], flim[1] + 0.25, 0.25)                       # MHz axis (ascending)
    ts = (t - t[0]).total_seconds().to_numpy()
    dur = ts[-1]
    # fundamental lower branch 28 -> 18 MHz (F band); harmonic (x2) 56 -> 36 MHz stays in-band
    fL = 28 - 8 * (ts / dur) - 2.0 * (ts / dur) ** 2
    fU = fL * 1.09                                                     # 9% band split -> X = 1.19
    layer = np.full((f.size, t.size), 0.15)
    def paint(track, s):
        for j, tv in enumerate(track * s):
            k = np.argmin(np.abs(f - tv))
            lo, hi = max(0, k - 3), min(f.size, k + 4)
            layer[lo:hi, j] += np.exp(-0.5 * ((f[lo:hi] - tv) / 0.6) ** 2)
    for tr in (fL, fU):
        paint(tr, 1)                                                  # F band
        paint(tr, 2)                                                  # H band (2x frequency)
    layer += np.random.default_rng(1).normal(0, 0.03, layer.shape)
    return t, f, np.clip(layer, 1e-3, None), dict(fL=fL, fU=fU, ts=ts)


if DEMO_MODE:
    t2_t, t2_f, t2_layer, _synth = synth_typeii_layer(TYPEII_WINDOW, TYPEII_FLIM)
    print(f'DEMO synthetic layer: {t2_layer.shape[0]} channels x {t2_layer.shape[1]} times, '
          f'{t2_f[0]:.1f}-{t2_f[-1]:.1f} MHz')
else:
    lwa_t, lwa_f, lwa_s = load_lwa_dyspec(LWA_FITS, LWA_POL_PLANE)
    print(f'{len(lwa_f)} channels, {len(lwa_t)} times, {lwa_f[0]:.1f}-{lwa_f[-1]:.1f} MHz')
    t2_t, t2_f, t2_layer = make_typeii_layer(lwa_t, lwa_f, lwa_s,
                                             TYPEII_WINDOW, TYPEII_FLIM, RADIO_BKG_WINDOW)
    _synth = None

NT, NF = t2_layer.shape[1], t2_layer.shape[0]
print(f'tracing array (freq, time) = {t2_layer.shape}')

In [ ]:

def draw_lwa_layer(ax, t=t2_t, f=t2_f, layer=t2_layer, index_axes=False,
                   vlo=35, vhi=99, cmap='RdYlBu_r'):
    """Draw the tracing layer with pcolormesh (down-sampled in time for speed). If index_axes,
    the axes are in pixel-index space (for Bezier placement) with UT/MHz tick labels; otherwise
    physical UT x MHz axes. Returns the QuadMesh."""
    step = max(1, layer.shape[1] // PREVIEW_MAX_TCOLS)
    vmin = np.nanpercentile(layer, vlo)
    vmax = np.nanpercentile(layer, vhi)
    norm = colors.LogNorm(vmin=vmin, vmax=vmax)
    if index_axes:
        nf, nt = layer.shape
        xcol = np.arange(nt)[::step]
        pm = ax.pcolormesh(xcol, np.arange(nf), layer[:, ::step], norm=norm, cmap=cmap,
                           rasterized=True)
        xt = np.linspace(0, nt - 1, 4).astype(int)
        ax.set_xticks(xt)
        ax.set_xticklabels([pd.Timestamp(t[i]).strftime('%H:%M:%S') for i in xt])
        yt = np.linspace(0, nf - 1, 6).astype(int)
        ax.set_yticks(yt)
        ax.set_yticklabels([f'{f[i]:.0f}' for i in yt])
    else:
        pm = ax.pcolormesh(t[::step], f, layer[:, ::step], norm=norm, cmap=cmap, rasterized=True)
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    ax.set_xlabel('Time [UT]')
    ax.set_ylabel('Frequency [MHz]')
    return pm


fig, ax = plt.subplots(figsize=[12, 6])
pm = draw_lwa_layer(ax)
fig.colorbar(pm, ax=ax, pad=0.01, label='ratio to pre-flare background')
ax.set_title('OVRO-LWA type II tracing window' + ('  (synthetic)' if DEMO_MODE else ''))
save_fig(fig, 'lwa_typeii_window')
plt.show()


## 2&#160;&#183;&#160;Electron-density models &#215; fold numbers

Each model returns $n_e(r)$ in cm$^{-3}$ for heliocentric distance $r$ in solar radii, scaled by a
**fold** factor (density enhancement over the quiet Sun). An observed frequency maps to a local
density $n_e=(f/(s\,f_1))^2$ with $s=1$ (F) or $2$ (H), which each model inverts to a source height.
Density jump $X$ and $M_A$ are **model-independent**; height, speed, $v_A$ and $B$ are not.

In [ ]:

def newkirk(r, fold=1):
    """Newkirk (1961), exponential quiet-corona model."""
    r = np.asarray(r, float)
    return fold * 4.2e4 * 10.0 ** (4.32 / r)

def saito(r, fold=1):
    """Saito (1970), equatorial two-term fit."""
    r = np.asarray(r, float)
    return fold * (1.36e6 * r ** -2.14 + 1.68e8 * r ** -6.13)

def leblanc(r, fold=1):
    """Leblanc, Dulk & Bougeret (1998), low corona to 1 AU."""
    r = np.asarray(r, float)
    return fold * (3.3e5 * r ** -2 + 4.1e6 * r ** -4 + 8.0e7 * r ** -6)

def baumbach_allen(r, fold=1):
    """Baumbach (1937) eclipse fit with Allen (1947) dust correction, three-term."""
    r = np.asarray(r, float)
    return fold * 1e8 * (0.036 * r ** -1.5 + 1.55 * r ** -6 + 2.99 * r ** -16)

def mann2023(r, fold=1):
    """Mann, Warmuth, Vocks & Rouillard (2023), A&A 679, A64, Eq. 18 (barometric, PSP-calibrated;
    valid r <~ 3 Rsun). n_e = 7.17e8 * exp(11.14 * (1/r - 1)). The constant 11.14 (their text)
    reproduces the paper's quoted n_e(3 Rsun) = 4.267e5 cm^-3; the printed Eq. 18 shows 11.35."""
    r = np.asarray(r, float)
    return fold * 7.17e8 * np.exp(11.14 * (1.0 / r - 1.0))

BASE_MODELS = {'Newkirk': newkirk, 'Saito': saito, 'Leblanc': leblanc,
               'Baumbach-Allen': baumbach_allen, 'Mann 2023': mann2023}

# every model x fold, bound as a single-argument callable model(r)
MODEL_GRID = {f'{name} x{fold}': (lambda r, _f=fun, _k=fold: _f(r, fold=_k))
              for name, fun in BASE_MODELS.items() for fold in FOLDS}
print(f'{len(BASE_MODELS)} models x {len(FOLDS)} folds = {len(MODEL_GRID)} combinations')


def freq_to_density(f_hz, harmonic=1):
    """Plasma-frequency -> local electron density [cm^-3] for emission at the given harmonic."""
    return (np.asarray(f_hz, float) / harmonic / PLASMA_CONST) ** 2

def freq_to_radius(f_hz, model, harmonic=1, r_bounds=(1.0, 5.0), nres=6000):
    """Invert a monotonically decreasing n_e(r) for the source height (vectorised);
    out-of-range densities -> NaN."""
    rr = np.linspace(r_bounds[0], r_bounds[1], nres)
    ne = model(rr)
    ne_t = freq_to_density(np.atleast_1d(f_hz), harmonic=harmonic)
    r = np.interp(ne_t, ne[::-1], rr[::-1])
    r[(ne_t > np.nanmax(ne)) | (ne_t < np.nanmin(ne))] = np.nan
    return r if r.size > 1 else float(r[0])

def alfven_mach_from_X(X, gamma=5/3):
    """Perpendicular-shock M_A from the density jump X (Rankine-Hugoniot); valid 1 <= X < 4."""
    X = np.asarray(X, float)
    out = np.full(X.shape, np.nan)
    ok = (X >= 1) & (X < 4)
    out[ok] = np.sqrt(X[ok] * (X[ok] + 5) / (2 * (4 - X[ok])))
    return out

In [ ]:

# n_e(r) grid and the frequency-height mapping the tracing relies on
rr = np.linspace(1.0, 3.0, 400)
fig, (axn, axf) = plt.subplots(1, 2, figsize=[13, 5])
for name, fun in BASE_MODELS.items():
    axn.plot(rr, fun(rr, fold=1), label=name)
axn.set_yscale('log')
axn.set_xlabel(r'$r\,/\,R_\odot$')
axn.set_ylabel(r'$n_e$ [cm$^{-3}$]')
axn.set_title('(a) Density models (fold 1)')
axn.legend(fontsize=8)
axn.grid(alpha=0.3)

ref = MODEL_GRID[REF_MODEL_NAME]
for s, lab in [(1, 'F (s=1)'), (2, 'H (s=2)')]:
    axf.plot(rr, s * PLASMA_CONST * np.sqrt(ref(rr)) / 1e6, label=lab)
axf.axhspan(TYPEII_FLIM[0], TYPEII_FLIM[1], color='0.85', label='tracing band')
axf.set_xlabel(r'$r\,/\,R_\odot$')
axf.set_ylabel(r'$f_{\rm em}$ [MHz]')
axf.set_title(f'(b) Emission frequency ({REF_MODEL_NAME})')
axf.legend(fontsize=8, loc='upper right')
axf.grid(alpha=0.3)
fig.tight_layout()
save_fig(fig, 'density_models_grid')
plt.show()


## 3&#160;&#183;&#160;Trace the band-split lanes

Two interchangeable methods populate the same `passes` structure &#8212; set `TRACE_MODE`:

- **`'bezier'`** &#8212; place a B&#233;zier curve per lane with sliders (or type an exact value in the
  box beside each slider). A quadratic has one anchor, a cubic two (`BEZIER_ANCHORS`). Each committed
  curve is expanded into a **jittered ensemble** (control points perturbed by `BEZIER_JITTER` px),
  so the SEM reflects placement sensitivity. Moving a slider updates only the curve and control
  points (blitted) &#8212; the spectrogram is drawn once.
- **`'manual'`** &#8212; the classic point-and-click picker: left-click to add points, right-click to
  finish a rep, `N_REPS_MANUAL` reps per lane.

`LANES_TO_TRACE` controls which of the four lanes (F/H &#215; lower/upper) are traced; the rest are
skipped and simply carry no diagnostics.

In [ ]:

def draw_bezier(x1=0, y1=0, x2=0, y2=0, controls=[[0, 0]], n=2, num_points=30):
    """Bezier curve of degree n from (x1,y1) to (x2,y2) with n-1 control points.
    n=1 linear, n=2 quadratic (1 anchor), n=3 cubic (2 anchors). Returns (num_points, 2)."""
    P0 = np.array([x1, y1], dtype=float)
    P3 = np.array([x2, y2], dtype=float)
    t = np.linspace(0, 1, num_points)
    if n == 1:
        curve = (1 - t)[:, None] * P0 + t[:, None] * P3
    elif n == 2:
        P1 = np.array(controls[0], dtype=float)
        curve = ((1 - t)[:, None] ** 2 * P0
                 + 2 * (1 - t)[:, None] * t[:, None] * P1
                 + t[:, None] ** 2 * P3)
    elif n == 3:
        P1 = np.array(controls[0], dtype=float)
        P2 = np.array(controls[1], dtype=float)
        curve = ((1 - t)[:, None] ** 3 * P0
                 + 3 * (1 - t)[:, None] ** 2 * t[:, None] * P1
                 + 3 * (1 - t)[:, None] * t[:, None] ** 2 * P2
                 + t[:, None] ** 3 * P3)
    else:
        raise ValueError('n must be 1 (linear), 2 (quadratic) or 3 (cubic).')
    return curve


def extract_bezier_values(array, x1, y1, x2, y2, controls, n, num_points):
    """Sample `array` (indexed [y, x]) along the Bezier curve; returns (values, x_idx, y_idx)
    as integer pixel indices clipped to the array bounds."""
    curve = draw_bezier(x1, y1, x2, y2, controls, n, num_points)
    x_coords = np.clip(np.round(curve[:, 0]).astype(int), 0, array.shape[1] - 1)
    y_coords = np.clip(np.round(curve[:, 1]).astype(int), 0, array.shape[0] - 1)
    values = array[y_coords, x_coords]
    return values, x_coords, y_coords


def bezier_freq_lane(times, freqs, x_idx, y_idx):
    """Map integer (time-idx, freq-idx) curve samples to a {'t':[...], 'f':[...]} lane
    (frequencies in MHz), sorted in time, duplicate times collapsed."""
    t_sel = pd.to_datetime([times[i] for i in x_idx])
    f_val = np.asarray(freqs, float)[y_idx]
    x_num = mdates.date2num(t_sel)
    order = np.argsort(x_num)
    t_sorted = np.asarray(t_sel)[order]
    f_sorted = f_val[order]
    xn = x_num[order]
    _, uniq = np.unique(xn, return_index=True)          # strictly increasing time for the fits
    return {'t': list(t_sorted[uniq]), 'f': list(f_sorted[uniq])}


def make_bezier_passes(specs, times, freqs, array, n_repeats=BEZIER_N_REPEATS,
                       jitter=BEZIER_JITTER, num_points=BEZIER_NUM_POINTS, seed=BEZIER_SEED):
    """Build the `passes` list (pas[band][branch] = {'t','f'}) from per-lane Bezier specs.
    specs[(band,branch)] = dict(p_start=(x,y), p_end=(x,y), anchors=[(x,y), ...]) in index space.
    The k-th jittered curve of every lane forms pass k, so lanes stay coherent within a pass."""
    rng = np.random.default_rng(seed)
    lanes = [('F', 'lower'), ('F', 'upper'), ('H', 'lower'), ('H', 'upper')]
    per_lane = {}
    for ln in lanes:
        sp = specs.get(ln)
        if sp is None:
            per_lane[ln] = None
            continue
        base = np.array([sp['p_start'], sp['p_end'], *sp['anchors']], float)
        n = len(sp['anchors']) + 1
        reps = []
        for r in range(n_repeats):
            pert = base if (r == 0 or jitter <= 0) else base + rng.normal(0, jitter, base.shape)
            (sx, sy), (ex, ey) = pert[0], pert[1]
            ctrl = [list(p) for p in pert[2:]]
            _, xi, yi = extract_bezier_values(array, sx, sy, ex, ey, ctrl, n, num_points)
            reps.append(bezier_freq_lane(times, freqs, xi, yi))
        per_lane[ln] = reps
    passes = []
    for k in range(n_repeats):
        pas = {b: {p: {'t': [], 'f': []} for p in ('lower', 'upper')} for b in ('F', 'H')}
        for ln in lanes:
            if per_lane[ln] is not None:
                pas[ln[0]][ln[1]] = per_lane[ln][k]
        passes.append(pas)
    n_traced = sum(v is not None for v in per_lane.values())
    print(f'built {len(passes)} passes from {n_traced} traced lane(s)')
    return passes

In [ ]:

_LIVE = {}                       # (band, branch) -> latest slider state
bezier_specs = {}                # committed curves used to build the passes

class LaneTracer:
    """Efficient interactive Bezier placement for one lane. The spectrogram is drawn ONCE with
    pcolormesh; moving a slider (or typing in its number box) updates only the curve and the
    control points via blitting, so the mesh is never re-rendered. The Bezier is placed in
    dynamic-spectrum index space (x = time frame, y = frequency channel)."""

    def __init__(self, band, branch):
        import ipywidgets as widgets
        from ipywidgets import IntSlider, BoundedIntText, HBox, VBox, Label, Layout, jslink
        from IPython.display import display
        self.key = (band, branch)
        self.band = band
        self.branch = branch
        self.n_anchors = BEZIER_ANCHORS[self.key]
        self.Nf, self.Nt = t2_layer.shape

        plt.ioff()
        self.fig, self.ax = plt.subplots(figsize=[11, 5])
        plt.ion()
        try:
            self.fig.canvas.header_visible = False
        except Exception:
            pass
        draw_lwa_layer(self.ax, index_axes=True)
        kind = 'quadratic (1 anchor)' if self.n_anchors == 1 else 'cubic (2 anchors)'
        self.ax.set_title(f'Place {band}-{branch}  [{kind}]  -  move sliders / type a value, then commit')

        # animated overlay artists: black curve, white-with-black-outline endpoints and anchors
        (self.curve_line,) = self.ax.plot([], [], '-', color='k', lw=2.2, animated=True)
        (self.end_pts,) = self.ax.plot([], [], 'o', mfc='white', mec='black', mew=1.5,
                                       ms=9, ls='none', animated=True)
        (self.anchor_pts,) = self.ax.plot([], [], '^', mfc='white', mec='black', mew=1.3,
                                          ms=10, ls='none', animated=True)
        self._bg = None
        self.fig.canvas.mpl_connect('draw_event', self._on_draw)

        d = self._defaults()
        specs = [('x_start', 'start x (time)', 0, self.Nt - 1),
                 ('y_start', 'start y (freq)', 0, self.Nf - 1),
                 ('x_end',   'end x (time)',   0, self.Nt - 1),
                 ('y_end',   'end y (freq)',   0, self.Nf - 1),
                 ('cx1',     'anchor1 x',      0, self.Nt - 1),
                 ('cy1',     'anchor1 y',      0, self.Nf - 1)]
        if self.n_anchors == 2:
            specs += [('cx2', 'anchor2 x', 0, self.Nt - 1),
                      ('cy2', 'anchor2 y', 0, self.Nf - 1)]
        self._sliders = {}
        rows = []
        for name, label, lo, hi in specs:
            sld = IntSlider(value=d[name], min=lo, max=hi, step=1, readout=False,
                            continuous_update=True, layout=Layout(width='560px'))
            box = BoundedIntText(value=d[name], min=lo, max=hi, step=1, layout=Layout(width='95px'))
            jslink((sld, 'value'), (box, 'value'))
            sld.observe(self._update, names='value')
            self._sliders[name] = sld
            rows.append(HBox([Label(label, layout=Layout(width='115px')), sld, box]))
        self.controls = VBox(rows)
        display(VBox([self.controls, self.fig.canvas]))
        self._update()

    def _defaults(self):
        base_y = 0.62 if self.branch == 'lower' else 0.74
        if self.band == 'H':
            base_y = min(0.95, base_y + 0.15)
        return dict(x_start=int(0.10 * self.Nt), y_start=int(base_y * self.Nf),
                    x_end=int(0.90 * self.Nt), y_end=int((base_y - 0.30) * self.Nf),
                    cx1=int(0.50 * self.Nt), cy1=int((base_y - 0.10) * self.Nf),
                    cx2=int(0.70 * self.Nt), cy2=int((base_y - 0.22) * self.Nf))

    def _vals(self):
        return {k: s.value for k, s in self._sliders.items()}

    def _anchors(self, v):
        if self.n_anchors == 1:
            return [(v['cx1'], v['cy1'])]
        return [(v['cx1'], v['cy1']), (v['cx2'], v['cy2'])]

    def _on_draw(self, event=None):
        self._bg = self.fig.canvas.copy_from_bbox(self.ax.bbox)
        self._blit()

    def _blit(self):
        self.ax.draw_artist(self.curve_line)
        self.ax.draw_artist(self.end_pts)
        self.ax.draw_artist(self.anchor_pts)

    def _update(self, change=None):
        v = self._vals()
        anchors = self._anchors(v)
        curve = draw_bezier(v['x_start'], v['y_start'], v['x_end'], v['y_end'],
                            anchors, self.n_anchors + 1, BEZIER_NUM_POINTS)
        self.curve_line.set_data(curve[:, 0], curve[:, 1])
        self.end_pts.set_data([v['x_start'], v['x_end']], [v['y_start'], v['y_end']])
        self.anchor_pts.set_data([a[0] for a in anchors], [a[1] for a in anchors])
        _LIVE[self.key] = dict(p_start=(v['x_start'], v['y_start']),
                               p_end=(v['x_end'], v['y_end']), anchors=anchors)
        if self._bg is not None:
            self.fig.canvas.restore_region(self._bg)
            self._blit()
            self.fig.canvas.blit(self.ax.bbox)
        else:
            self.fig.canvas.draw_idle()


def commit_lanes():
    """Copy the latest slider state of every placed, non-skipped lane into bezier_specs."""
    for k, v in _LIVE.items():
        if k in LANES_TO_TRACE:
            bezier_specs[k] = dict(p_start=tuple(v['p_start']), p_end=tuple(v['p_end']),
                                   anchors=[tuple(a) for a in v['anchors']])
    print('committed lanes:', ', '.join(sorted(f'{b}-{p}' for (b, p) in bezier_specs)) or '(none)')

In [ ]:

class TypeIIPicker:
    """Classic point-and-click lane tracer (blitted). Left-click adds a point to the active lane's
    current rep, right-click finishes the rep; N_REPS_MANUAL reps per lane. `passes` assembles the
    i-th rep of every lane into pass i, matching the Bezier output."""

    STYLE = {('F', 'lower'): ('navy', 'o'), ('F', 'upper'): ('red', 'o'),
             ('H', 'lower'): ('darkcyan', 's'), ('H', 'upper'): ('magenta', 's')}

    def __init__(self, t_index, freqs, layer, lanes=None, n_reps=5, flim=(16, 62)):
        import ipywidgets as widgets
        from IPython.display import display
        self.t_index = t_index
        self.freqs = freqs
        self.layer = layer
        self.LANES = list(lanes) if lanes else [('F', 'lower'), ('F', 'upper'),
                                                ('H', 'lower'), ('H', 'upper')]
        self.n_reps = n_reps
        self.flim = flim
        self.reps = {lane: [] for lane in self.LANES}
        self.current = {'t': [], 'f': []}
        self.active = self.LANES[0]
        self._build(widgets, display)

    @property
    def passes(self):
        out = []
        for i in range(self.n_reps):
            pas = {b: {ln: {'t': [], 'f': []} for ln in ('lower', 'upper')} for b in ('F', 'H')}
            for (b, ln), reps in self.reps.items():
                if i < len(reps):
                    pas[b][ln] = {'t': list(reps[i]['t']), 'f': list(reps[i]['f'])}
            if any(pas[b][ln]['f'] for b in ('F', 'H') for ln in ('lower', 'upper')):
                out.append(pas)
        return out

    def _build(self, widgets, display):
        plt.ioff()
        self.fig, self.ax = plt.subplots(figsize=[11, 5.5])
        plt.ion()
        try:
            self.fig.canvas.header_visible = False
        except Exception:
            pass
        draw_lwa_layer(self.ax, t=self.t_index, f=self.freqs, layer=self.layer, index_axes=False)
        self.ax.set_ylim(*self.flim)
        self.ax.set_xlim(self.t_index[0], self.t_index[-1])
        self.ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
        self.scat = {}
        for lane in self.LANES:
            c, m = self.STYLE[lane]
            (self.scat[lane],) = self.ax.plot([], [], m, color=c, ms=7, mew=1.4, mfc='none',
                                              animated=True, label=f'{lane[0]}-{lane[1]}')
        (self.ghost,) = self.ax.plot([], [], '.', color='0.5', ms=3, alpha=0.4, animated=True)
        self.ax.legend(loc='upper right', fontsize=8, ncol=2)
        self._bg = None
        self.fig.canvas.mpl_connect('draw_event', self._on_draw)
        self._title()
        self.sel = widgets.ToggleButtons(options=[f'{b}-{l}' for b, l in self.LANES],
                                         value=f'{self.active[0]}-{self.active[1]}', description='Lane:')
        self.undo_btn = widgets.Button(description='Undo', icon='rotate-left')
        self.rep_btn = widgets.Button(description='Finish rep', icon='check', button_style='success')
        self.status = widgets.HTML()
        self.sel.observe(self._on_sel, names='value')
        self.undo_btn.on_click(lambda _: self._undo())
        self.rep_btn.on_click(lambda _: self._finish_rep())
        self.fig.canvas.mpl_connect('button_press_event', self._on_click)
        display(widgets.VBox([widgets.HBox([self.sel, self.undo_btn, self.rep_btn]),
                              self.status, self.fig.canvas]))
        self._update_status()

    def _title(self):
        b, l = self.active
        done = len(self.reps[self.active])
        rep_no = min(done + 1, self.n_reps)
        state = 'COMPLETE - switch lane' if done >= self.n_reps else f'rep {rep_no} / {self.n_reps}'
        self.ax.set_title(f'{b}-{l}:  {state}   -   left-click: add  |  right-click: finish rep')

    def _on_sel(self, change):
        self.active = tuple(change['new'].split('-'))
        self.current = {'t': [], 'f': []}
        self._title()
        self.fig.canvas.draw_idle()
        self._update_status()

    def _on_click(self, event):
        if event.inaxes != self.ax or event.xdata is None:
            return
        if event.button == 3:
            self._finish_rep()
            return
        if len(self.reps[self.active]) >= self.n_reps:
            self._update_status(f'{self.active[0]}-{self.active[1]} already has '
                                f'{self.n_reps} reps - switch lane.', warn=True)
            return
        t = mdates.num2date(event.xdata).replace(tzinfo=None)
        self.current['t'].append(pd.Timestamp(t))
        self.current['f'].append(float(event.ydata))
        self._redraw()
        self._update_status()

    def _on_draw(self, event=None):
        self._bg = self.fig.canvas.copy_from_bbox(self.ax.bbox)
        self._blit_markers()

    def _blit_markers(self):
        for lane in self.LANES:
            self.ax.draw_artist(self.scat[lane])
        self.ax.draw_artist(self.ghost)

    def _redraw(self):
        for lane in self.LANES:
            if lane == self.active and self.current['f']:
                x = mdates.date2num(pd.to_datetime(self.current['t']))
                self.scat[lane].set_data(x, self.current['f'])
            else:
                self.scat[lane].set_data([], [])
        if self._bg is not None:
            self.fig.canvas.restore_region(self._bg)
            self._blit_markers()
            self.fig.canvas.blit(self.ax.bbox)
        else:
            self.fig.canvas.draw_idle()

    def _undo(self):
        if self.current['t']:
            self.current['t'].pop()
            self.current['f'].pop()
            self._redraw()
            self._update_status()

    def _refresh_ghost(self):
        gx, gy = [], []
        for lane in self.LANES:
            for rep in self.reps[lane]:
                gx += list(mdates.date2num(pd.to_datetime(rep['t'])))
                gy += list(rep['f'])
        self.ghost.set_data(gx, gy)

    def _finish_rep(self):
        lane = self.active
        if len(self.reps[lane]) >= self.n_reps:
            self._update_status(f'{lane[0]}-{lane[1]}: {self.n_reps} reps already done.', warn=True)
            return
        if len(self.current['f']) == 0:
            self._update_status('Nothing traced in this rep yet.', warn=True)
            return
        self.reps[lane].append({'t': list(self.current['t']), 'f': list(self.current['f'])})
        self.current = {'t': [], 'f': []}
        self._refresh_ghost()
        done = len(self.reps[lane])
        self._title()
        self.fig.canvas.draw_idle()
        self._update_status(f'{lane[0]}-{lane[1]}: rep {done}/{self.n_reps} recorded.')

    def _update_status(self, msg='', warn=False):
        counts = ' | '.join(f'{b}-{l}:{len(self.reps[(b, l)])}/{self.n_reps}'
                            for (b, l) in self.LANES)
        color = '#a00' if warn else '#060'
        self.status.value = (f'<b>Reps per lane:</b> {counts} &nbsp; '
                             f'<b>current rep:</b> {len(self.current["f"])} pts '
                             f'&nbsp; <span style="color:{color}">{msg}</span>')

**Bezier mode.** Run the four cells below (one per lane). Adjust the sliders or type an exact
index in the box beside each slider; the black curve and white control points update live. Skipped
lanes (not in `LANES_TO_TRACE`) print a note instead. When all wanted lanes are placed, run the
**commit** cell. (Ignored in manual/demo mode.)

In [ ]:
%matplotlib widget
if TRACE_MODE == 'bezier' and not DEMO_MODE and ('F', 'lower') in LANES_TO_TRACE:
    tracer_F_lower = LaneTracer('F', 'lower')
else:
    print("skipped: not bezier mode, DEMO_MODE, or ('F','lower') not in LANES_TO_TRACE")

In [ ]:
if TRACE_MODE == 'bezier' and not DEMO_MODE and ('F', 'upper') in LANES_TO_TRACE:
    tracer_F_upper = LaneTracer('F', 'upper')
else:
    print("skipped: not bezier mode, DEMO_MODE, or ('F','upper') not in LANES_TO_TRACE")

In [ ]:
if TRACE_MODE == 'bezier' and not DEMO_MODE and ('H', 'lower') in LANES_TO_TRACE:
    tracer_H_lower = LaneTracer('H', 'lower')
else:
    print("skipped: not bezier mode, DEMO_MODE, or ('H','lower') not in LANES_TO_TRACE")

In [ ]:
if TRACE_MODE == 'bezier' and not DEMO_MODE and ('H', 'upper') in LANES_TO_TRACE:
    tracer_H_upper = LaneTracer('H', 'upper')
else:
    print("skipped: not bezier mode, DEMO_MODE, or ('H','upper') not in LANES_TO_TRACE")

**Manual mode.** Run this cell to launch the classic point-and-click picker (ignored in
bezier/demo mode).

In [ ]:
%matplotlib widget
if TRACE_MODE == 'manual' and not DEMO_MODE:
    picker = TypeIIPicker(t2_t, t2_f, t2_layer, lanes=LANES_TO_TRACE,
                          n_reps=N_REPS_MANUAL, flim=TYPEII_FLIM)
else:
    print('manual picker skipped (bezier mode or DEMO_MODE)')

In [ ]:

# ---- build the passes from whichever method is active (or fabricate them under DEMO_MODE) ----
def _demo_specs():
    """Index-space Bezier specs straight from the synthetic tracks (bypasses the widget)."""
    ts, fL, fU = _synth['ts'], _synth['fL'], _synth['fU']
    def idx(track_mhz):
        xs = np.linspace(0, NT - 1, 5).astype(int)
        pts = []
        for x in xs:
            j = int(round(x / (NT - 1) * (len(ts) - 1)))
            pts.append((int(x), int(np.argmin(np.abs(t2_f - track_mhz[j])))))
        return pts
    tracks = {('F', 'lower'): fL, ('F', 'upper'): fU, ('H', 'lower'): 2 * fL, ('H', 'upper'): 2 * fU}
    specs = {}
    for lane in LANES_TO_TRACE:
        pts = idx(tracks[lane])
        na = BEZIER_ANCHORS[lane]
        anchors = [pts[2]] if na == 1 else [pts[1], pts[3]]
        specs[lane] = dict(p_start=pts[0], p_end=pts[-1], anchors=anchors)
    return specs

if DEMO_MODE:
    bezier_specs = _demo_specs()
    print('DEMO_MODE synthetic specs for:', ', '.join(f'{b}-{p}' for (b, p) in bezier_specs))
    passes = make_bezier_passes(bezier_specs, t2_t, t2_f, t2_layer)
elif TRACE_MODE == 'bezier':
    commit_lanes()
    passes = make_bezier_passes(bezier_specs, t2_t, t2_f, t2_layer)
else:
    passes = picker.passes
    print(f'{len(passes)} passes from the point-and-click picker')

In [ ]:

# preview the traced lanes on the (down-sampled) dynamic spectrum -- works for either method
LANE_COL = {('F', 'lower'): 'navy', ('F', 'upper'): 'lime',
            ('H', 'lower'): 'cyan', ('H', 'upper'): 'magenta'}
fig, ax = plt.subplots(figsize=[12, 6])
pm = draw_lwa_layer(ax)
fig.colorbar(pm, ax=ax, pad=0.01, label='ratio to pre-flare background')
for lane in LANES_TO_TRACE:
    b, br = lane
    for k, pas in enumerate(passes):
        L = pas[b][br]
        if L['f']:
            ax.plot(pd.to_datetime(L['t']), L['f'], color=LANE_COL[lane],
                    lw=(2.0 if k == 0 else 0.6), alpha=(1.0 if k == 0 else 0.3),
                    label=(f'{b}-{br}' if k == 0 else None))
ax.set_title('Traced lanes (bold = reference pass, thin = ensemble / reps)')
ax.legend(fontsize=8, ncol=2, loc='upper right')
save_fig(fig, 'traced_lanes_preview')
plt.show()


## 4&#160;&#183;&#160;Shock diagnostics from the traced lanes

Per pass the lanes are fitted with quadratics $f(t)$. Per band the density jump is $X=(f_U/f_L)^2$
and $M_A=\sqrt{X(X+5)/[2(4-X)]}$. Each branch frequency maps to $n_e=(f/(s f_1))^2$ and, via the
chosen model, to a height $r(t)$; shock speed and acceleration are $\dot r$ and $\ddot r$. The
Alfv&#233;n speed is $v_A=v_{\rm sh}/M_A$ and $B=v_A\sqrt{\mu_0\rho}$ with $\rho=\mu m_p n_e$. The
Monte-Carlo loop resamples each lane polynomial from its covariance for every pass, giving the
combined statistical uncertainty ($\pm$SEM).

In [ ]:

t0 = TYPEII_WINDOW[0]                     # shared time origin for all passes

def _lane_fit(lane, deg=2):
    """Quadratic fit of a lane f(t), keeping the coefficient covariance for the fit error."""
    t = pd.to_datetime(lane['t'])
    f = np.asarray(lane['f'], float)
    if len(f) < 2:
        return None
    ts = (t - t0).total_seconds().to_numpy()
    o = np.argsort(ts)
    ts, f = ts[o], f[o]
    d = min(deg, len(ts) - 1)
    if len(ts) >= d + 2:
        p, cov = np.polyfit(ts, f, d, cov=True)
    else:
        p, cov = np.polyfit(ts, f, d), None
    return {'p': p, 'cov': cov, 'tmin': ts.min(), 'tmax': ts.max()}

def _coeffs(fit, rng=None, sample=False):
    if fit is None:
        return None
    if sample and rng is not None and fit['cov'] is not None:
        try:
            return rng.multivariate_normal(fit['p'], fit['cov'])
        except Exception:
            return fit['p']
    return fit['p']

def _eval(fit, coeffs, tg):
    if fit is None or coeffs is None:
        return np.full_like(tg, np.nan)
    f = np.polyval(coeffs, tg)
    f[(tg < fit['tmin']) | (tg > fit['tmax'])] = np.nan
    return f

def pass_fits(pas):
    return {b: {ln: _lane_fit(pas[b][ln]) for ln in ('lower', 'upper')} for b in ('F', 'H')}

def build_grid(passes, n=40):
    secs = []
    for pas in passes:
        for b in ('F', 'H'):
            for l in ('lower', 'upper'):
                L = pas[b][l]
                if L['f']:
                    secs += list((pd.to_datetime(L['t']) - t0).total_seconds())
    secs = np.asarray(secs, float)
    return np.linspace(np.nanmin(secs), np.nanmax(secs), n)

def _invert_grid(model, r_bounds=(1.0, 5.0), nres=6000):
    rr = np.linspace(r_bounds[0], r_bounds[1], nres)
    return rr, model(rr)

def sg_smooth(y, window=9, poly=2):
    y = np.asarray(y, float)
    m = np.isfinite(y)
    if m.sum() < 5:
        return y
    yy = y.copy()
    yy[~m] = np.interp(np.flatnonzero(~m), np.flatnonzero(m), y[m])
    w = min(window, len(yy))
    if w % 2 == 0:
        w -= 1
    if w < poly + 2:
        return yy
    return savgol_filter(yy, w, poly)

In [ ]:

def realise_lanes(fits, tg, invert, rng=None, sample=False, mu=MU):
    """One realisation of per-lane height/speed/accel/vA/B for all four lanes. Height, v and a use
    each lane's own frequency (F: s=1, H: s=2). B additionally needs the band's density jump
    X=(f_U/f_L)^2, so it is only defined where both split branches of that band are traced."""
    rr, ne = invert
    ne_min, ne_max = np.nanmin(ne), np.nanmax(ne)
    coeff = {(b, p): _coeffs(fits[b][p], rng, sample)
             for b in ('F', 'H') for p in ('lower', 'upper')}
    fval = {(b, p): _eval(fits[b][p], coeff[(b, p)], tg)
            for b in ('F', 'H') for p in ('lower', 'upper')}
    out = {}
    for b in ('F', 'H'):
        s = HARM[b]
        with np.errstate(all='ignore'):
            X = (fval[(b, 'upper')] / fval[(b, 'lower')]) ** 2
        MA = alfven_mach_from_X(X)
        for p in ('lower', 'upper'):
            ne_t = freq_to_density(fval[(b, p)] * 1e6, harmonic=s)
            r = np.interp(ne_t, ne[::-1], rr[::-1])
            r[(ne_t > ne_max) | (ne_t < ne_min)] = np.nan
            v = np.full_like(tg, np.nan)
            a = np.full_like(tg, np.nan)
            good = np.isfinite(r)
            frac = good.sum() / good.size
            r_range = (np.nanmax(r) - np.nanmin(r)) if good.sum() > 3 else 0.0
            if good.sum() > 3 and frac > 0.4 and r_range > 0.01:
                pr = np.polyfit(tg[good], r[good], min(3, good.sum() - 1))
                v = np.polyval(np.polyder(pr, 1), tg) * R_SUN_M / 1e3     # km/s
                a = np.polyval(np.polyder(pr, 2), tg) * R_SUN_M           # m/s^2
            v_phys = np.where((v > 0) & (v < 3000), v, np.nan)
            vA = v_phys / MA                                             # km/s
            rho = mu * M_P * (ne_t * 1e6)                                # kg/m^3
            B = (vA * 1e3) * np.sqrt(MU0 * rho) * 1e4                    # Gauss
            out[(b, p)] = dict(r=r, v=v, a=a, vA=vA, B=B, X=X, MA=MA, ne=ne_t)
    return out

def aggregate_lanes(passes, tg, model, n_mc=N_MC, seed=0):
    """Per-lane Monte-Carlo aggregation (combined fitting + repeat error)."""
    rng = np.random.default_rng(seed)
    invert = _invert_grid(model)
    fitsets = [pass_fits(pas) for pas in passes]
    keys = ('r', 'v', 'a', 'vA', 'B', 'X', 'MA', 'ne')
    lanes = [(b, p) for b in ('F', 'H') for p in ('lower', 'upper')]
    stacks = {ln: {k: [] for k in keys} for ln in lanes}
    for fits in fitsets:
        for _ in range(n_mc):
            res = realise_lanes(fits, tg, invert, rng=rng, sample=True)
            for ln in lanes:
                for k in keys:
                    stacks[ln][k].append(res[ln][k])
    out = {}
    npass = len(passes)
    for ln in lanes:
        out[ln] = {}
        for k in keys:
            M = np.vstack(stacks[ln][k])
            out[ln][k + '_mean'] = np.nanmean(M, axis=0)
            out[ln][k + '_sd'] = np.nanstd(M, axis=0)
            out[ln][k + '_se'] = out[ln][k + '_sd'] / np.sqrt(npass)
    return out


def scalar_summary(passes):
    """Band-independent scalars: density jump X, M_A, drift rate and relative bandwidth,
    each as mean +/- standard error across the passes. Drift is reported per band."""
    rows = {}
    for b in ('F', 'H'):
        Xv, MAv, drift, relbw = [], [], [], []
        for pas in passes:
            fl = _lane_fit(pas[b]['lower'])
            fu = _lane_fit(pas[b]['upper'])
            if fl and fu:
                lo, hi = max(fl['tmin'], fu['tmin']), min(fl['tmax'], fu['tmax'])
                if hi > lo:
                    gg = np.linspace(lo, hi, 20)
                    fLv, fUv = np.polyval(fl['p'], gg), np.polyval(fu['p'], gg)
                    Xg = (fUv / fLv) ** 2
                    Xv.append(np.nanmean(Xg))
                    MAv.append(np.nanmean(alfven_mach_from_X(Xg)))
                    relbw.append(np.nanmean((fUv - fLv) / fLv))
            if fl:
                gg = np.linspace(fl['tmin'], fl['tmax'], 20)
                drift.append(np.nanmean(np.polyval(np.polyder(fl['p']), gg)))   # MHz/s
        def mse(arr):
            arr = np.asarray(arr, float)
            if arr.size == 0:
                return (np.nan, np.nan)
            se = np.nanstd(arr, ddof=1) / np.sqrt(len(arr)) if len(arr) > 1 else 0.0
            return np.nanmean(arr), se
        rows[b] = {'X': mse(Xv), 'M_A': mse(MAv), 'drift_MHz_s': mse(drift),
                   'rel_bandwidth': mse(relbw)}
    return rows


def electron_energy_from_speed(v_kms):
    """Kinetic energy [keV] of an electron at speed v: E=(gamma-1) m_e c^2. For a type II this is
    the exciter-frame electron energy at the shock speed (non-relativistic here; the relativistic
    form is kept so the same estimator serves fast type III beams)."""
    v = np.asarray(v_kms, float) * 1e3
    beta = np.clip(v / C_MS, 0, 0.999999)
    gamma = 1.0 / np.sqrt(1.0 - beta ** 2)
    return (gamma - 1.0) * M_E * C_MS ** 2 / E_CHARGE_J / 1e3

In [ ]:

tg = build_grid(passes)
ALref = aggregate_lanes(passes, tg, MODEL_GRID[REF_MODEL_NAME])
scalars = scalar_summary(passes)

print(f'Reference model: {REF_MODEL_NAME}\n')
for b in ('F', 'H'):
    s = scalars[b]
    if np.isfinite(s['X'][0]):
        print(f"  {b} band:  drift = {s['drift_MHz_s'][0]:+.3f} +/- {s['drift_MHz_s'][1]:.3f} MHz/s,  "
              f"X = {s['X'][0]:.2f} +/- {s['X'][1]:.2f},  M_A = {s['M_A'][0]:.2f} +/- {s['M_A'][1]:.2f}")
for lane in [('F', 'lower'), ('H', 'lower')]:
    d = ALref[lane]
    if np.isfinite(d['r_mean']).sum() > 2:
        E = electron_energy_from_speed(np.nanmean(d['v_mean']))
        print(f"  {lane[0]}-{lane[1]}:  r = {np.nanmean(d['r_mean']):.3f} Rsun,  "
              f"v_sh = {np.nanmean(d['v_mean']):.0f} +/- {np.nanmean(d['v_se']):.0f} km/s,  "
              f"v_A = {np.nanmean(d['vA_mean']):.0f} km/s,  B = {np.nanmean(d['B_mean']):.2f} G,  "
              f"E_exc = {E*1e3:.2f} eV")


## 5&#160;&#183;&#160;Per-lane kinematics and coronal magnetic field

Height, shock speed, acceleration and $B$ for every traced lane on the reference model, with the
Monte-Carlo $\pm$SEM error bars (points) and a Savitzky&#8211;Golay smooth (line).

In [ ]:

LANES4   = [('F', 'lower'), ('F', 'upper'), ('H', 'lower'), ('H', 'upper')]
LANE_KCOL = {('F', 'lower'): 'navy', ('F', 'upper'): 'firebrick',
             ('H', 'lower'): 'darkcyan', ('H', 'upper'): 'magenta'}
AL = ALref

def _traced(lane, key):
    return np.isfinite(AL[lane][key + '_mean']).sum() > 2

fig, axs = plt.subplots(2, 2, figsize=[13, 9])
(axr, axv), (axa, axb) = axs

def lane_panel(ax, key, ylabel, unit, fmt='{:.3g}'):
    for ln in LANES4:
        if ln not in LANES_TO_TRACE or not _traced(ln, key):
            continue
        c = LANE_KCOL[ln]
        m, err = AL[ln][key + '_mean'], AL[ln][key + '_se']
        mbar = np.nanmean(m)
        sebar = np.sqrt(np.nanmean(err ** 2))
        ax.errorbar(tg, m, yerr=err, fmt='o', ms=3, color=c, ecolor=c,
                    elinewidth=0.7, capsize=1.5, alpha=0.45)
        ax.plot(tg, sg_smooth(m), '-', color=c, lw=1.8,
                label=f'{ln[0]}-{ln[1]}: {fmt.format(mbar)} $\\pm$ {fmt.format(sebar)} {unit}')
    ax.set_ylabel(ylabel)
    ax.set_xlabel(f'time since {t0.strftime("%H:%M")} UT [s]')
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

lane_panel(axr, 'r', r'$r\,/\,R_\odot$', r'$R_\odot$', '{:.3f}')
axr.set_title(f'(a) Height-time  ({REF_MODEL_NAME})')
lane_panel(axv, 'v', r'$v_{\rm sh}$ [km s$^{-1}$]', 'km/s', '{:.0f}')
axv.set_title('(b) Shock speed')
lane_panel(axa, 'a', r'$a$ [m s$^{-2}$]', r'm s$^{-2}$', '{:.0f}')
axa.axhline(0, color='0.6', lw=0.8)
axa.set_title('(c) Acceleration')
lane_panel(axb, 'B', r'$B$ [G]', 'G', '{:.2f}')
axb.set_title('(d) Coronal magnetic field')

fig.suptitle(f'Type II per-lane kinematics and coronal B  ({EVENT_DATE})', y=1.01, fontsize=13)
fig.tight_layout()
save_fig(fig, 'typeii_lane_kinematics_Bfield')
plt.show()


## 5b&#160;&#183;&#160;Height-time fitting: polynomial vs Gallagher vs Byrne

The reference-lane height-time (from the reference density model) is fit three ways and the speed
$v=\dot h$ and acceleration $a=\ddot h$ compared, with bootstrap error bands:

- **Polynomial** &#8212; degree `POLY_DEG`, analytic derivatives.
- **Gallagher (2003)** &#8212; the reciprocal-sum acceleration profile
  $a(t)=[1/(a_r e^{t/\tau_r})+1/(a_d e^{-t/\tau_d})]^{-1}$, with $h(t)$ the **exact double integral**
  of $a(t)$ (not the $h_0+v_0t+\tfrac12 a t^2$ shortcut).
- **Byrne (2013)** &#8212; the Savitzky&#8211;Golay-with-bootstrapping method (this is the physical
  Byrne approach; the analytic $\arctan$ form sometimes attributed to Byrne 2013 is not in that
  paper and is dimensionally inhomogeneous, so it is deliberately not used here).

In [ ]:

RS_KM = R_SUN_M / 1e3

def pick_ref_lane(agg):
    order = [('F', 'lower'), ('H', 'lower'), ('F', 'upper'), ('H', 'upper')]
    for ln in order:
        if ln in LANES_TO_TRACE and np.isfinite(agg[ln]['r_mean']).sum() > 3:
            return ln
    for ln in order:
        if np.isfinite(agg[ln]['r_mean']).sum() > 3:
            return ln
    return ('F', 'lower')

ref_lane = pick_ref_lane(ALref)
_d = ALref[ref_lane]
_good = np.isfinite(_d['r_mean'])
t_fit = tg[_good].astype(float)
r_fit = _d['r_mean'][_good].astype(float)
h_fit = r_fit * RS_KM                                   # km
se_r = _d['r_se'][_good].astype(float)
_pos = se_r[np.isfinite(se_r) & (se_r > 0)]
se_r = np.where(np.isfinite(se_r) & (se_r > 0), se_r, (np.nanmedian(_pos) if _pos.size else 1e-3))
sig_h = se_r * RS_KM
t_dense = np.linspace(t_fit.min(), t_fit.max(), 400)


def fit_polynomial(t, h, sig):
    p = np.polyfit(t, h, POLY_DEG, w=(1.0 / sig if sig is not None else None))
    return lambda tt: np.polyval(p, tt)

def gallagher_accel(t, ar, ad, tr, td):
    """Gallagher, Lawrence & Dennis (2003) reciprocal-sum acceleration profile a(t) [km/s^2]."""
    return 1.0 / (1.0 / (ar * np.exp(t / tr)) + 1.0 / (ad * np.exp(-t / td)))

def fit_gallagher(t, h, sig):
    tgrid = np.linspace(t.min(), t.max(), 400)
    def model(tt, ar, ad, tr, td, h0, v0):
        a = gallagher_accel(tgrid, ar, ad, tr, td)
        v = v0 + cumulative_trapezoid(a, tgrid, initial=0)      # exact single integral
        hh = h0 + cumulative_trapezoid(v, tgrid, initial=0)     # exact double integral
        return np.interp(tt, tgrid, hh)
    v0g = (h[-1] - h[0]) / (t[-1] - t[0])
    p0 = [1e-3, 1e-3, 150, 150, h[0], v0g]
    lo = [1e-6, 1e-6, 20, 20, h[0] - 5e4, -2000]
    hi = [1e2, 1e2, 5e3, 5e3, h[0] + 5e4, 3000]
    popt, _ = curve_fit(model, t, h, p0=p0, sigma=sig, absolute_sigma=False,
                        bounds=(lo, hi), maxfev=200000)
    return lambda tt: model(tt, *popt)

def fit_byrne(t, h, sig):
    """Byrne et al. (2013): Savitzky-Golay smoothing of h(t) (bootstrapped for the error band)."""
    n = len(h)
    win = min(11, n if n % 2 == 1 else n - 1)
    if win <= POLY_DEG:
        win = POLY_DEG + 2 if (POLY_DEG + 2) % 2 == 1 else POLY_DEG + 3
    hs = savgol_filter(h, win, POLY_DEG)
    return lambda tt: np.interp(tt, t, hs)

def hva(hfun, t, dt=DT_DERIV):
    hp, h0, hm = hfun(t + dt), hfun(t), hfun(t - dt)
    v = (hp - hm) / (2 * dt)                             # km/s
    a = (hp - 2 * h0 + hm) / dt ** 2 * 1000              # m/s^2
    return h0, v, a

FIT_METHODS = {'Polynomial': fit_polynomial, 'Gallagher (2003)': fit_gallagher,
               'Byrne (2013)': fit_byrne}
FIT_COLOR = {'Polynomial': 'tab:blue', 'Gallagher (2003)': 'tab:green', 'Byrne (2013)': 'tab:red'}

fit_out = {}
rng = np.random.default_rng(1)
for name, fitter in FIT_METHODS.items():
    try:
        f0 = fitter(t_fit, h_fit, sig_h)
    except Exception as ex:
        print(f'{name} fit failed: {ex}')
        continue
    h0, v0, a0 = hva(f0, t_dense)
    boot = {'h': [], 'v': [], 'a': []}
    for _ in tqdm(range(N_BOOT), desc=name):
        hb = h_fit + rng.normal(0, sig_h)
        try:
            fb = fitter(t_fit, hb, sig_h)
            hh, vv, aa = hva(fb, t_dense)
            boot['h'].append(hh)
            boot['v'].append(vv)
            boot['a'].append(aa)
        except Exception:
            continue
    fit_out[name] = dict(
        h=h0 / RS_KM, v=v0, a=a0,
        h_sd=(np.nanstd(boot['h'], axis=0) / RS_KM if boot['h'] else np.zeros_like(t_dense)),
        v_sd=(np.nanstd(boot['v'], axis=0) if boot['v'] else np.zeros_like(t_dense)),
        a_sd=(np.nanstd(boot['a'], axis=0) if boot['a'] else np.zeros_like(t_dense)))
print(f'reference lane for the height-time fits: {ref_lane[0]}-{ref_lane[1]}')

In [ ]:

fig, axs = plt.subplots(1, 3, figsize=[16, 4.6])
(axh, axv2, axa2) = axs
axh.errorbar(t_fit, r_fit, yerr=se_r, fmt='o', ms=4, color='0.4', capsize=2,
             alpha=0.7, label='traced (mean $\\pm$ SEM)', zorder=1)
for name, d in fit_out.items():
    c = FIT_COLOR[name]
    axh.plot(t_dense, d['h'], '-', color=c, lw=1.8, label=name)
    axh.fill_between(t_dense, d['h'] - d['h_sd'], d['h'] + d['h_sd'], color=c, alpha=0.18)
    axv2.plot(t_dense, d['v'], '-', color=c, lw=1.8, label=name)
    axv2.fill_between(t_dense, d['v'] - d['v_sd'], d['v'] + d['v_sd'], color=c, alpha=0.18)
    axa2.plot(t_dense, d['a'], '-', color=c, lw=1.8, label=name)
    axa2.fill_between(t_dense, d['a'] - d['a_sd'], d['a'] + d['a_sd'], color=c, alpha=0.18)
for ax, ylab, ttl in [(axh, r'$r\,/\,R_\odot$', '(a) Height'),
                      (axv2, r'$v_{\rm sh}$ [km s$^{-1}$]', '(b) Speed'),
                      (axa2, r'$a$ [m s$^{-2}$]', '(c) Acceleration')]:
    ax.set_xlabel(f'time since {t0.strftime("%H:%M")} UT [s]')
    ax.set_ylabel(ylab)
    ax.set_title(ttl)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
axa2.axhline(0, color='0.6', lw=0.8)
fig.suptitle(f'Height-time fit comparison, lane {ref_lane[0]}-{ref_lane[1]}  ({REF_MODEL_NAME})',
             y=1.03, fontsize=13)
fig.tight_layout()
save_fig(fig, 'typeii_kinematics_fit_comparison')
plt.show()


## 6&#160;&#183;&#160;Sensitivity to density model &#215; fold

The band-split **$X$ and $M_A$ are model-independent** (quoted once, &#167;8). The **height, shock
speed, Alfv&#233;n speed and $B$** depend on the density model &#8212; each is grid-averaged over the
burst for all $5\times4$ combinations with Monte-Carlo error bars. The spread across the grid is the
density-model systematic, which dominates the statistical bars.

In [ ]:

def grid_scalar(d, key):
    m, se = d[key + '_mean'], d[key + '_se']
    good = np.isfinite(m)
    if good.sum() == 0:
        return np.nan, np.nan
    return np.nanmean(m[good]), np.sqrt(np.nanmean(se[good] ** 2))

sweep_rows = []
for name, model in tqdm(MODEL_GRID.items(), desc='model x fold'):
    agg = aggregate_lanes(passes, tg, model)
    d = agg[ref_lane]
    row = {'model': name}
    for key, lab in [('r', 'r_Rsun'), ('v', 'v_kms'), ('vA', 'vA_kms'), ('B', 'B_G'), ('ne', 'ne_cm3')]:
        row[lab], row[lab + '_se'] = grid_scalar(d, key)
    sweep_rows.append(row)
sweep = pd.DataFrame(sweep_rows)
sweep['base'] = [m.rsplit(' x', 1)[0] for m in sweep['model']]
sweep['fold'] = [int(m.rsplit(' x', 1)[1]) for m in sweep['model']]
sweep.to_csv(os.path.join(OUTDIR, 'model_grid_sweep.csv'), index=False)
print(f'reference lane for the sweep: {ref_lane[0]}-{ref_lane[1]}')
sweep[['model', 'r_Rsun', 'v_kms', 'vA_kms', 'B_G']].round(3)

In [ ]:

bases = list(BASE_MODELS.keys())
xpos = np.arange(len(bases))
fold_off = {f: (f - 2.5) * 0.16 for f in FOLDS}
fold_col = {1: 'tab:blue', 2: 'tab:orange', 3: 'tab:green', 4: 'tab:red'}

panels = [('r_Rsun', r'height $r$ [$R_\odot$]'), ('v_kms', r'shock speed $v_{\rm sh}$ [km s$^{-1}$]'),
          ('vA_kms', r'Alfven speed $v_A$ [km s$^{-1}$]'), ('B_G', r'$B$ [G]')]
fig, axs = plt.subplots(2, 2, figsize=[13, 9])
for ax, (col, ylab) in zip(axs.ravel(), panels):
    for f in FOLDS:
        sub = sweep[sweep['fold'] == f].set_index('base').reindex(bases)
        ax.errorbar(xpos + fold_off[f], sub[col], yerr=sub[col + '_se'], fmt='o',
                    color=fold_col[f], ms=6, capsize=3, label=f'fold {f}')
    ax.set_xticks(xpos)
    ax.set_xticklabels(bases, fontsize=8)
    ax.set_ylabel(ylab)
    ax.grid(alpha=0.3, axis='y')
axs[0, 0].legend(title='fold', fontsize=8, ncol=2)
fig.suptitle(f'Shock characteristics vs density model x fold  (lane {ref_lane[0]}-{ref_lane[1]})',
             y=1.01, fontsize=13)
fig.tight_layout()
save_fig(fig, 'characteristics_vs_model_fold')
plt.show()


## 7&#160;&#183;&#160;Coronal magnetic field vs empirical $B(r)$ laws

The band-split estimate $B=v_A\sqrt{\mu_0\rho}$ (one point per model&#215;fold at the height that
model assigns, with error bars) is compared with Dulk & McLean (1978), Gopalswamy & Yashiro (2011,
calibrated 6&#8211;23 $R_\odot$ so extrapolated here), and Mann et al. (2023, Eq. 8).

In [ ]:

def B_dulk_mclean(r):
    """Dulk & McLean (1978); valid 1.02 <= r <= 10 Rsun."""
    r = np.asarray(r, float)
    return 0.5 * (r - 1.0) ** -1.5

def B_gopalswamy_yashiro(r):
    """Gopalswamy & Yashiro (2011) headline fit (calibrated 6-23 Rsun; extrapolated below)."""
    r = np.asarray(r, float)
    return 0.377 * r ** -1.25

def B_mann2023(r):
    """Mann et al. (2023), A&A 679, A64, Eq. 8 radial field [G]."""
    r = np.asarray(r, float)
    return 6.0 * r ** -3 + 1.18 * r ** -2

rr = np.linspace(1.05, 2.5, 300)
fig, ax = plt.subplots(figsize=[9, 6])
ax.plot(rr, B_dulk_mclean(rr), 'k-', label='Dulk & McLean (1978)')
ax.plot(rr, B_gopalswamy_yashiro(rr), 'k--', label='Gopalswamy & Yashiro (2011), extrapolated')
ax.plot(rr, B_mann2023(rr), 'k:', label='Mann et al. (2023)')

mk = {'Newkirk': 'o', 'Saito': 's', 'Leblanc': '^', 'Baumbach-Allen': 'D', 'Mann 2023': 'v'}
for _, row in sweep.iterrows():
    if np.isfinite(row['r_Rsun']) and np.isfinite(row['B_G']):
        ax.errorbar(row['r_Rsun'], row['B_G'], yerr=row['B_G_se'], xerr=row['r_Rsun_se'],
                    fmt=mk[row['base']], color=fold_col[row['fold']], ms=8, capsize=2,
                    mec='k', mew=0.5, alpha=0.9)
handles = [plt.Line2D([], [], marker=mk[b], color='0.4', ls='', mec='k', label=b) for b in bases]
handles += [plt.Line2D([], [], marker='o', color=fold_col[f], ls='', label=f'fold {f}') for f in FOLDS]
leg1 = ax.legend(loc='upper right', fontsize=9)
ax.add_artist(leg1)
ax.legend(handles=handles, loc='lower left', fontsize=8, ncol=2, title='band-split estimate')
ax.set_yscale('log')
ax.set_xlabel(r'$r\,/\,R_\odot$')
ax.set_ylabel(r'$B$ [G]')
ax.set_title('Coronal magnetic field: band-split estimate vs empirical laws')
ax.grid(alpha=0.3, which='both')
fig.tight_layout()
save_fig(fig, 'Bfield_comparison')
plt.show()

## 8&#160;&#183;&#160;Characteristics table and saved picks

In [ ]:

rows = []
for b in ('F', 'H'):
    s = scalars[b]
    rows.append({'quantity': f'drift rate ({b})', 'value': s['drift_MHz_s'][0],
                 'error': s['drift_MHz_s'][1], 'unit': 'MHz/s', 'note': 'model-independent'})
    rows.append({'quantity': f'density jump X ({b})', 'value': s['X'][0], 'error': s['X'][1],
                 'unit': '-', 'note': 'model-independent'})
    rows.append({'quantity': f'Alfven Mach number M_A ({b})', 'value': s['M_A'][0], 'error': s['M_A'][1],
                 'unit': '-', 'note': 'model-independent'})
    rows.append({'quantity': f'relative bandwidth ({b})', 'value': s['rel_bandwidth'][0],
                 'error': s['rel_bandwidth'][1], 'unit': '-', 'note': 'model-independent'})
for lane in LANES4:
    if lane not in LANES_TO_TRACE:
        continue
    d = ALref[lane]
    if np.isfinite(d['r_mean']).sum() <= 2:
        continue
    tag = f'{lane[0]}-{lane[1]}'
    for key, lab, unit in [('r', 'height r', 'Rsun'), ('v', 'shock speed', 'km/s'),
                           ('a', 'acceleration', 'm/s^2'), ('vA', 'Alfven speed', 'km/s'),
                           ('B', 'magnetic field B', 'G')]:
        m, se = grid_scalar(d, key)
        rows.append({'quantity': f'{lab} ({tag})', 'value': m, 'error': se, 'unit': unit,
                     'note': REF_MODEL_NAME})
    v_mean, _ = grid_scalar(d, 'v')
    rows.append({'quantity': f'exciter e- energy ({tag})',
                 'value': electron_energy_from_speed(v_mean) * 1e3, 'error': np.nan,
                 'unit': 'eV', 'note': f'{REF_MODEL_NAME}, E=(gamma-1)m_e c^2 at v_sh'})

char_table = pd.DataFrame(rows)
char_table.to_csv(os.path.join(OUTDIR, 'typeii_characteristics.csv'), index=False)

picks = {'bezier_specs': bezier_specs, 'passes': passes,
         'config': {'TRACE_MODE': TRACE_MODE, 'TYPEII_WINDOW': TYPEII_WINDOW,
                    'TYPEII_FLIM': TYPEII_FLIM, 'HARM': HARM, 'REF_MODEL_NAME': REF_MODEL_NAME,
                    'LANES_TO_TRACE': LANES_TO_TRACE, 'BEZIER_ANCHORS': BEZIER_ANCHORS}}
with open(os.path.join(OUTDIR, 'typeii_bezier_picks.pkl'), 'wb') as fh:
    pickle.dump(picks, fh)
print('saved typeii_characteristics.csv, model_grid_sweep.csv, typeii_bezier_picks.pkl to', OUTDIR)
char_table.round(3)


### Conventions and caveats

- **Tracing method** is set by `TRACE_MODE` (`'bezier'` or `'manual'`); both feed the identical
  `passes` structure and downstream physics. `LANES_TO_TRACE` skips lanes.
- **Model-independent vs model-dependent.** $X$, $M_A$, drift rate and bandwidth depend only on the
  traced frequencies; height, speed, $v_A$ and $B$ depend on the density model &#8212; that spread
  (&#167;6) is the real uncertainty budget, larger than the statistical bars.
- **Height-time fits (&#167;5b).** Byrne is the Savitzky&#8211;Golay + bootstrap method; the analytic
  $\arctan$ form sometimes labelled "Byrne 2013" is not in that paper and is not used.
- **Exciter electron energy** is $(\gamma-1)m_ec^2$ at the shock speed &#8212; the bulk exciter
  energy, not a relativistic beam.
- **$M_A$ validity** needs $1\le X<4$; band splits with $X\ge4$ return NaN. **Mann (2023)** density is
  valid to $\sim3\,R_\odot$; Gopalswamy & Yashiro (2011) $B(r)$ is extrapolated below 6&#8211;23
  $R_\odot$.